# OBLITERATUS — One-Click Abliteration

**SOTA refusal removal** running on free Colab GPU. SVD multi-direction extraction, norm-preserving projection, iterative refinement.

Based on: Arditi et al. (2024) | Gabliteration (arXiv:2512.18901) | grimjim norm-preserving biprojection (2025)

---

**How to use:**
1. Make sure GPU runtime is enabled: `Runtime > Change runtime type > T4 GPU`
2. Set your model and method in the config cell below
3. Run All (`Runtime > Run all` or `Ctrl+F9`)
4. Download the abliterated model from the output

## 1. Install

In [1]:
!pip install -q git+https://github.com/elder-plinius/OBLITERATUS.git
!pip install -q accelerate bitsandbytes

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. Configure

Edit the cell below to set your target model and abliteration method.

In [2]:
#@title Abliteration Config { run: "auto" }

#@markdown ### Target Model
#@markdown Pick a model from the dropdown or paste a custom HuggingFace ID.
MODEL = "meta-llama/Llama-3.1-8B-Instruct" #@param ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "microsoft/Phi-3.5-mini-instruct", "THUDM/glm-4-9b-chat", "NousResearch/Hermes-3-Llama-3.1-8B", "cognitivecomputations/dolphin-2.9.4-llama3.1-8b", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "openai-community/gpt2"] {allow-input: true}

#@markdown ### Method
METHOD = "advanced" #@param ["basic", "advanced", "aggressive"]

#@markdown ### Advanced Overrides (leave 0 to use method defaults)
N_DIRECTIONS = 0 #@param {type: "integer"}
REGULARIZATION = 0.0 #@param {type: "number"}
REFINEMENT_PASSES = 0 #@param {type: "integer"}

#@markdown ### Output
OUTPUT_DIR = "abliterated" #@param {type: "string"}

print(f"Model: {MODEL}")
print(f"Method: {METHOD}")
print(f"Output: {OUTPUT_DIR}/")

Model: meta-llama/Llama-3.1-8B-Instruct
Method: advanced
Output: abliterated/


## 3. Run Abliteration Pipeline

This runs all 6 stages: SUMMON → PROBE → DISTILL → EXCISE → VERIFY → REBIRTH

In [5]:
from obliteratus.abliterate import AbliterationPipeline

# Build kwargs, only pass overrides if non-zero
kwargs = dict(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="auto",
    dtype="float16",
    method=METHOD,
)
if N_DIRECTIONS > 0:
    kwargs["n_directions"] = N_DIRECTIONS
if REGULARIZATION > 0:
    kwargs["regularization"] = REGULARIZATION
if REFINEMENT_PASSES > 0:
    kwargs["refinement_passes"] = REFINEMENT_PASSES

# Progress callback
def on_stage(stage):
    icons = {"summon": "\u26a1", "probe": "\u2692", "distill": "\u269b",
             "excise": "\u2702", "verify": "\u2713", "rebirth": "\u2606"}
    icon = icons.get(stage.stage, "")
    print(f"\n{'='*60}")
    print(f"{icon}  STAGE: {stage.stage.upper()} \u2014 {stage.message}")
    print(f"{'='*60}")

def on_log(msg):
    print(f"  {msg}")

kwargs["on_stage"] = on_stage
kwargs["on_log"] = on_log

pipeline = AbliterationPipeline(**kwargs)
result = pipeline.run()

print(f"\n{'='*60}")
print(f"ABLITERATION COMPLETE")
print(f"Output: {result.get('output_dir', OUTPUT_DIR)}")
print(f"{'='*60}")


⚡  STAGE: SUMMON — Loading meta-llama/Llama-3.1-8B-Instruct...
  Loading model: meta-llama/Llama-3.1-8B-Instruct
  Device: auto | Dtype: float16
  Method: Advanced (Multi-direction + Norm-preserving)
    Directions: 4 (svd) | Norm-preserve: True
    Regularization: 0.3 | Refinement passes: 2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RuntimeError: Access denied for gated model 'meta-llama/Llama-3.1-8B-Instruct'.

This model requires you to:
  1. Accept the license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
  2. Set your HF_TOKEN: export HF_TOKEN=hf_...
     (or add it to your HF Space secrets)

Token is NOT currently set.

## 4. Download the Abliterated Model

Run the cell below to zip and download, or upload directly to HuggingFace Hub.

In [ ]:
import os
from pathlib import Path

# Find the output directory
out_path = Path(OUTPUT_DIR)
subdirs = [d for d in out_path.iterdir() if d.is_dir()] if out_path.exists() else []
model_dir = subdirs[0] if subdirs else out_path

print(f"Model saved at: {model_dir}")
print(f"Contents:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

In [ ]:
#@title Option A: Download as ZIP
import shutil
from google.colab import files

zip_name = model_dir.name
shutil.make_archive(zip_name, 'zip', model_dir)
print(f"Downloading {zip_name}.zip ...")
files.download(f"{zip_name}.zip")

In [ ]:
#@title Option B: Push to HuggingFace Hub
#@markdown Set your HF repo name. You'll need to be logged in (`huggingface-cli login`).
HF_REPO = "your-username/model-name-abliterated" #@param {type: "string"}

from huggingface_hub import HfApi
api = HfApi()

# Login if needed
from huggingface_hub import notebook_login
notebook_login()

# Upload
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=str(model_dir),
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"\nUploaded to: https://huggingface.co/{HF_REPO}")

## 5. Test the Abliterated Model

Quick sanity check — try some prompts that the original model would refuse.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading abliterated model from {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    torch_dtype=torch.float16,
    device_map="auto",
)

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "The capital of France is",
    "Explain how a neural network works:",
    "Write a poem about the ocean:",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    print(generate(prompt))